# 3. 검토 (Validation)

**A. 데이터 출처 검증** — 생성된 `data/*.parquet` 3종이 원천 `cache/els3_dataset.parquet`(및 raw)과 일치하는지 직접 비교.
**B. 결과 일치 검증** — 새 파이프라인 `result/statistics/metrics.csv` 가 기존 `ELS_3star_benchmark.ipynb`(→ `scratch/benchmark_reference.csv`) 결과와 일치하는지 비교.

> 원천 체인: raw(`data/raw/LAKE_V2_*.csv`) → [scratch/exp15,17] → `cache/els3_dataset.parquet` → [0_data] → `data/*.parquet`. 여기선 마지막 링크(data ↔ cache)를 값 단위로 검증.

## A. 데이터 출처 검증 (data/*.parquet ↔ cache/els3_dataset.parquet)

In [ ]:
import numpy as np, pandas as pd
from util import file_manager as fm

src = pd.read_parquet(fm.source()).sort_values("isu_ord").reset_index(drop=True)
ml = pd.read_parquet(fm.dataset("ml")).sort_values("isu_ord").reset_index(drop=True)
dn = (
    pd.read_parquet(fm.dataset("deeponet"))
    .sort_values("isu_ord")
    .reset_index(drop=True)
)
cv = (
    pd.read_parquet(fm.dataset("deeponet_curve"))
    .sort_values("isu_ord")
    .reset_index(drop=True)
)


def match(a, b, col):
    return bool(
        np.allclose(
            a[col].values.astype(float), b[col].values.astype(float), equal_nan=True
        )
    )


checks = []
checks.append(
    (
        "row counts equal (src=ml=deeponet=curve)",
        len(src) == len(ml) == len(dn) == len(cv),
    )
)
checks.append(
    (
        "isu_ord aligned  ml vs src",
        np.array_equal(ml["isu_ord"].values, src["isu_ord"].values),
    )
)
checks.append(
    (
        "isu_ord aligned  deeponet vs src",
        np.array_equal(dn["isu_ord"].values, src["isu_ord"].values),
    )
)
checks.append(
    (
        "isu_ord aligned  curve vs src",
        np.array_equal(cv["isu_ord"].values, src["isu_ord"].values),
    )
)
for col in ["fair", "mc", "recent_margin"]:
    checks.append((f"ml[{col}] == src", match(ml, src, col)))
for col in ["sig_eff", "r", "K", "coupon", "tenor"]:
    checks.append((f"deeponet[{col}] == src", match(dn, src, col)))
for col in ["u0", "u5", "u9", "sig_eff", "coupon"]:
    checks.append((f"curve[{col}] == src", match(cv, src, col)))
# 3종 간 fair/mc 상호 일치
checks.append(
    (
        "fair consistent across 3 datasets",
        match(ml, dn, "fair") and match(ml, cv, "fair"),
    )
)
checks.append(
    ("mc consistent across 3 datasets", match(ml, dn, "mc") and match(ml, cv, "mc"))
)

for name, ok in checks:
    print(("PASS" if ok else "FAIL"), "-", name)
print("\n==> ALL PASS" if all(ok for _, ok in checks) else "\n==> SOME FAILED")

PASS - row counts equal (src=ml=deeponet=curve)
PASS - isu_ord aligned  ml vs src
PASS - isu_ord aligned  deeponet vs src
PASS - isu_ord aligned  curve vs src
PASS - ml[fair] == src
PASS - ml[mc] == src
PASS - ml[recent_margin] == src
PASS - deeponet[sig_eff] == src
PASS - deeponet[r] == src
PASS - deeponet[K] == src
PASS - deeponet[coupon] == src
PASS - deeponet[tenor] == src
PASS - curve[u0] == src
PASS - curve[u5] == src
PASS - curve[u9] == src
PASS - curve[sig_eff] == src
PASS - curve[coupon] == src
PASS - fair consistent across 3 datasets
PASS - mc consistent across 3 datasets

==> ALL PASS


In [ ]:
# raw 원천 참고 (provenance 컨텍스트)
import pandas as pd
from util import file_manager as fm

for p in sorted(fm.RAW.glob("*.csv")):
    try:
        d = pd.read_csv(p, nrows=3)
        print(f"{p.name}: cols={len(d.columns)} sample={list(d.columns)[:6]}")
    except Exception as e:
        print(p.name, "read err", e)

LAKE_V2_DART_AUTO_CALL.csv: cols=40 sample=['ITEM_CD', 'ITEM_NM', 'ITEM_NM_KRX', 'ISU_ORG', 'ISU_DT', 'MAT_DT']
LAKE_V2_DART_SCHD_INFO.csv: cols=26 sample=['ITEM_CD', 'SCHD_TYPE', 'SEQ', 'EXER_DT', 'STRK_1', 'STRK_2']
LAKE_V2_DART_UDLY_INFO.csv: cols=9 sample=['ITEM_CD', 'UDLY_ID', 'UDLY_NM', 'FXD_DT', 'INI_PRC', 'CUR_CD']


## B. 결과 일치 검증 (result/statistics/metrics.csv ↔ ELS_3star_benchmark)

In [ ]:
import pandas as pd
from util import file_manager as fm

new = pd.read_csv(fm.stat("metrics"), index_col=0)
ref = pd.read_csv(
    fm.SCRATCH / "benchmark_reference.csv"
)  # ELS_3star_benchmark 결과 (exp25 저장)

# 새 예측명 ↔ 기존 벤치마크명 매핑 (겹치는 8개)
mapping = {
    "deeponet_hybrid": "DeepONet Hybrid (MLP,noPDE)",
    "pi_deeponet_hybrid": "PI-DeepONet Hybrid (BEST,autograd)",
    "deeponet_curve_direct": "DeepONet (direct)",
    "bench_gbm": "GBM (direct)",
    "bench_ridge": "Ridge (direct)",
    "bench_lgbm": "LightGBM (direct)",
    "bench_catboost": "CatBoost (direct)",
    "bench_xgboost": "XGBoost (direct)",
}
refm = ref.set_index("model")
rows = []
for nnew, nold in mapping.items():
    if nnew in new.index and nold in refm.index:
        r_new, r_old = new.loc[nnew, "R2"], refm.loc[nold, "R2"]
        m_new, m_old = new.loc[nnew, "MAE"], refm.loc[nold, "MAE"]
        rows.append(
            {
                "model": nnew,
                "R2_new": r_new,
                "R2_ref": r_old,
                "dR2": r_new - r_old,
                "MAE_new": m_new,
                "MAE_ref": m_old,
                "match": abs(r_new - r_old) < 0.01 and abs(m_new - m_old) < 0.002,
            }
        )
cmp = pd.DataFrame(rows)
print(cmp.round(4).to_string(index=False))
print(
    "\n==> ALL MATCH (|dR2|<0.01, |dMAE|<0.002)"
    if cmp["match"].all()
    else "\n==> MISMATCH 발생 — 확인 필요"
)
print("\n(신규 모델 — 기존 벤치마크에 없음: deeponet_direct, deeponet_curve_hybrid)")

                model  R2_new  R2_ref     dR2  MAE_new  MAE_ref  match
      deeponet_hybrid  0.7274  0.7274 -0.0000   0.0210   0.0210   True
   pi_deeponet_hybrid  0.7239  0.7239 -0.0000   0.0213   0.0213   True
deeponet_curve_direct  0.0558  0.0511  0.0047   0.0394   0.0394   True
            bench_gbm  0.2569  0.2569 -0.0000   0.0371   0.0371   True
          bench_ridge  0.5608  0.5608 -0.0000   0.0269   0.0269   True
           bench_lgbm  0.3109  0.3109 -0.0000   0.0353   0.0353   True
       bench_catboost  0.3155  0.3155 -0.0000   0.0357   0.0357   True
        bench_xgboost  0.3196  0.3196 -0.0000   0.0352   0.0352   True

==> ALL MATCH (|dR2|<0.01, |dMAE|<0.002)

(신규 모델 — 기존 벤치마크에 없음: deeponet_direct, deeponet_curve_hybrid)
